# 資料集切分與 DehazeDDPM 測試集產生

本 Notebook 會完成以下工作：

1. 以固定種子切分清晰圖（train/val/test = 280/40/80）
2. 確保同一張清晰圖對應的 `light/medium/heavy` 霧圖不會跨 split
3. 自 `cold_depth_metric_vitl_980` 將 `{id}_raw_depth_meter.npy` 複製到各 split 的 `gt` 旁，並在 manifest 紀錄 **β**（與 `synthesize_fog.ipynb` 的 `BETA_CONFIG` 一致）
4. 複製主資料集 split 結果
5. 額外建立 DehazeDDPM 專用測試集（從 test 中每個 ID 隨機抽一種霧等級，且 `gt_test`/`hazy_test` 檔名一致）
6. 輸出 manifest 與最終統計檢查

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import random
import shutil
from datetime import datetime

# ===== 參數區：可依需求調整 =====
SEED = 42  # 固定隨機種子，確保每次切分結果可重現

DATA_ROOT = Path(".")
GT_ROOT = DATA_ROOT / "cold_inputs"
FOG_ROOT = DATA_ROOT / "cold_fog_synthesized"
# 與 Depth-Anything-V2/synthesize_fog.ipynb 一致：{stem}_raw_depth_meter.npy
DEPTH_ROOT = DATA_ROOT / "cold_depth_metric_vitl_980"
FOG_LEVELS = ("light", "medium", "heavy")

# ASM 散射係數（與 synthesize_fog 的 BETA_CONFIG 對齊）
BETA_MAP = {
    "light": 0.06,
    "medium": 0.20,
    "heavy": 0.50,
}

SPLITS_ROOT = DATA_ROOT / "splits"
DEHAZED_TEST_ROOT = DATA_ROOT / "dehazeddpm_test"

N_TRAIN = 280
N_VAL = 40
N_TEST = 80
EXPECTED_TOTAL = N_TRAIN + N_VAL + N_TEST

VALID_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
CLEAN_OUTPUT_DIRS = True  # True 會先清空舊輸出資料夾

print("=== 參數設定 ===")
print(f"Seed: {SEED}")
print(f"GT root: {GT_ROOT}")
print(f"Fog root: {FOG_ROOT}")
print(f"Depth root: {DEPTH_ROOT}")
print(f"Beta map: {BETA_MAP}")
print(f"Expected GT count: {EXPECTED_TOTAL}")

=== 參數設定 ===
Seed: 42
GT root: cold_inputs
Fog root: cold_fog_synthesized
Depth root: cold_depth_metric_vitl_980
Beta map: {'light': 0.06, 'medium': 0.2, 'heavy': 0.5}
Expected GT count: 400


## 1) 掃描資料與配對檢查

這一段會做三件事：

- 掃描 `cold_inputs` 的清晰圖 ID
- 檢查 `light/medium/heavy` 三個資料夾是否都存在對應 ID
- 檢查 `cold_depth_metric_vitl_980` 是否存在 `{id}_raw_depth_meter.npy`（與霧合成腳本配對方式相同）
- 若資料不足或有缺漏，直接報錯停止

In [2]:
# 判斷是否為支援的影像格式

def _is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VALID_SUFFIXES


def _depth_npy_for_id(sample_id: str) -> Path:
    """與 synthesize_fog.collect_pairs 相同：{stem}_raw_depth_meter.npy"""
    return DEPTH_ROOT / f"{sample_id}_raw_depth_meter.npy"


# 以「去副檔名」作為 ID，收集成 {id: 檔案路徑}
def _collect_files_by_stem(directory: Path) -> dict[str, Path]:
    files = sorted([p for p in directory.iterdir() if _is_image(p)])
    mapping: dict[str, Path] = {}
    dup_stems: list[str] = []

    for f in files:
        stem = f.stem
        if stem in mapping:
            dup_stems.append(stem)
        else:
            mapping[stem] = f

    if dup_stems:
        example = ", ".join(sorted(set(dup_stems))[:10])
        raise ValueError(
            f"在 {directory} 發現重複 ID：{example}，請確認每個 ID 只對應一張圖。"
        )
    return mapping


# 1) 基本路徑檢查
if not GT_ROOT.exists():
    raise FileNotFoundError(f"找不到清晰圖資料夾：{GT_ROOT.resolve()}")
if not FOG_ROOT.exists():
    raise FileNotFoundError(f"找不到霧圖資料夾：{FOG_ROOT.resolve()}")
if not DEPTH_ROOT.exists():
    raise FileNotFoundError(f"找不到 depth 資料夾：{DEPTH_ROOT.resolve()}")

for level in FOG_LEVELS:
    level_dir = FOG_ROOT / level
    if not level_dir.exists():
        raise FileNotFoundError(f"缺少霧等級資料夾：{level_dir.resolve()}")

# 2) 掃描 GT 與三個霧等級
# gt_by_id: {id: gt_path}
# fog_by_level_and_id: {level: {id: fog_path}}
gt_by_id = _collect_files_by_stem(GT_ROOT)
fog_by_level_and_id: dict[str, dict[str, Path]] = {
    level: _collect_files_by_stem(FOG_ROOT / level)
    for level in FOG_LEVELS
}

# 3) 檢查每個 GT ID 在三個霧等級都存在
all_gt_ids = set(gt_by_id.keys())
missing_fog_by_level: dict[str, set[str]] = {}
extra_fog_by_level: dict[str, set[str]] = {}

for level in FOG_LEVELS:
    level_ids = set(fog_by_level_and_id[level].keys())
    missing = all_gt_ids - level_ids
    extra = level_ids - all_gt_ids
    if missing:
        missing_fog_by_level[level] = missing
    if extra:
        extra_fog_by_level[level] = extra

if missing_fog_by_level:
    debug = {lvl: sorted(list(ids))[:10] for lvl, ids in missing_fog_by_level.items()}
    raise ValueError(f"有 GT 缺少對應霧圖，範例：{debug}")

if extra_fog_by_level:
    debug = {lvl: sorted(list(ids))[:10] for lvl, ids in extra_fog_by_level.items()}
    raise ValueError(f"有霧圖找不到對應 GT，範例：{debug}")

# 4) depth npy：每個 GT id 需存在對應 raw_depth_meter.npy
missing_depth = sorted(sid for sid in all_gt_ids if not _depth_npy_for_id(sid).is_file())
if missing_depth:
    ex = ", ".join(missing_depth[:10])
    raise ValueError(
        f"以下 GT 在 {DEPTH_ROOT} 缺少 depth 檔（需 {{id}}_raw_depth_meter.npy），範例：{ex}"
    )


all_ids = sorted(all_gt_ids)
if len(all_ids) < EXPECTED_TOTAL:
    raise ValueError(f"清晰圖數量不足：目前 {len(all_ids)}，至少需要 {EXPECTED_TOTAL}。")
if len(all_ids) != EXPECTED_TOTAL:
    print(
        f"提醒：清晰圖共有 {len(all_ids)} 張（預期 {EXPECTED_TOTAL}）。"
        "後續會以固定種子打散後取前 400 張進行切分。"
    )

print("=== 掃描與配對檢查完成 ===")
print(f"清晰圖數量：{len(all_ids)}")
for level in FOG_LEVELS:
    print(f"霧圖（{level}）數量：{len(fog_by_level_and_id[level])}")
print(f"Depth npy（可配對）：{len(all_gt_ids)}（每張 GT 一個 _raw_depth_meter.npy）")

=== 掃描與配對檢查完成 ===
清晰圖數量：400
霧圖（light）數量：400
霧圖（medium）數量：400
霧圖（heavy）數量：400
Depth npy（可配對）：400（每張 GT 一個 _raw_depth_meter.npy）


## 2) 固定種子切分 train / val / test

- 使用 `SEED` 打散 ID
- 依序切成 280 / 40 / 80
- 驗證三個集合互斥（不重疊）

In [3]:
# 用固定種子打散 ID，確保可重現
rng = random.Random(SEED)
shuffled_ids = all_ids.copy()
rng.shuffle(shuffled_ids)

# 若資料超過 400 張，只取前 EXPECTED_TOTAL 張進行此次切分
selected_ids = shuffled_ids[:EXPECTED_TOTAL]

# 按照 280 / 40 / 80 切分
train_ids = selected_ids[:N_TRAIN]
val_ids = selected_ids[N_TRAIN:N_TRAIN + N_VAL]
test_ids = selected_ids[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]

# 基本數量檢查
assert len(train_ids) == N_TRAIN
assert len(val_ids) == N_VAL
assert len(test_ids) == N_TEST

# 互斥檢查：不能重疊
train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)
assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

print("=== 可重現切分完成 ===")
print(f"train IDs: {len(train_ids)}")
print(f"val IDs:   {len(val_ids)}")
print(f"test IDs:  {len(test_ids)}")
print(f"train 範例 ID: {train_ids[:5]}")
print(f"val 範例 ID:   {val_ids[:5]}")
print(f"test 範例 ID:  {test_ids[:5]}")

=== 可重現切分完成 ===
train IDs: 280
val IDs:   40
test IDs:  80
train 範例 ID: ['sd_20260218_0385', 'ggl_20260202_0087', 'pin_20260202_0054', 'sd_20260213_0300', 'sd_20260218_0382']
val 範例 ID:   ['sd_20260218_0396', 'ggl_20260202_0063', 'sdm_20260212_0287', 'sdm_20260204_0234', 'sd_20260215_0340']
test 範例 ID:  ['ggl_20260202_0100', 'sd_20260215_0317', 'sd_20260215_0325', 'ggl_20260202_0076', 'sd_20260215_0322']


## 3) 複製主資料集 split（train / val / test）

此段會建立以下結構並複製檔案：

- `data/splits/train/gt`（含清晰圖與同名的 `{id}_raw_depth_meter.npy`）
- `data/splits/train/hazy/{light,medium,heavy}`
- `data/splits/val/...`
- `data/splits/test/...`

重點：同一個 ID 的三種霧圖會跟著同一個 split；β 由霧等級對應 `BETA_MAP`（寫入 manifest）。

In [4]:
# 建立/清空輸出資料夾
def _ensure_clean_dir(path: Path, clean: bool = False) -> None:
    if clean and path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


# 複製單一檔案（保留檔案資訊）
def _copy_file(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


split_id_map = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
}

_ensure_clean_dir(SPLITS_ROOT, clean=CLEAN_OUTPUT_DIRS)

# split_manifest：紀錄每個 split 的來源/去向，方便追蹤
split_manifest: dict[str, object] = {
    "seed": SEED,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "input": {
        "gt_root": str(GT_ROOT),
        "fog_root": str(FOG_ROOT),
        "fog_levels": list(FOG_LEVELS),
        "depth_root": str(DEPTH_ROOT),
        "depth_npy_suffix": "_raw_depth_meter.npy",
        "beta_by_fog_level": dict(BETA_MAP),
    },
    "counts": {},
    "splits": {},
}

for split_name, ids in split_id_map.items():
    split_gt_dir = SPLITS_ROOT / split_name / "gt"
    split_hazy_root = SPLITS_ROOT / split_name / "hazy"

    split_manifest["splits"][split_name] = []

    for sample_id in ids:
        # 複製 GT 與對應 metric depth（npy 置於同一路徑 gt/）
        gt_src = gt_by_id[sample_id]
        gt_dst = split_gt_dir / gt_src.name
        _copy_file(gt_src, gt_dst)

        depth_src = _depth_npy_for_id(sample_id)
        depth_dst = split_gt_dir / depth_src.name
        _copy_file(depth_src, depth_dst)

        # 複製三種霧圖（同一 ID 必須留在同一 split）
        hazy_entries: dict[str, str] = {}
        for level in FOG_LEVELS:
            fog_src = fog_by_level_and_id[level][sample_id]
            fog_dst = split_hazy_root / level / fog_src.name
            _copy_file(fog_src, fog_dst)
            hazy_entries[level] = str(fog_dst)

        split_manifest["splits"][split_name].append(
            {
                "id": sample_id,
                "gt": str(gt_dst),
                "depth": str(depth_dst),
                "hazy": hazy_entries,
                "beta": {lvl: BETA_MAP[lvl] for lvl in FOG_LEVELS},
            }
        )

    split_manifest["counts"][split_name] = {
        "gt": len(ids),
        "depth_npy": len(ids),
        "hazy_total": len(ids) * len(FOG_LEVELS),
        "hazy_per_level": {level: len(ids) for level in FOG_LEVELS},
    }

print("=== 主資料集 split 複製完成 ===")
print(f"輸出位置：{SPLITS_ROOT}")
for split_name in ("train", "val", "test"):
    info = split_manifest["counts"][split_name]
    print(f"{split_name}: GT={info['gt']}, depth_npy={info['depth_npy']}, HazyTotal={info['hazy_total']}")

=== 主資料集 split 複製完成 ===
輸出位置：splits
train: GT=280, depth_npy=280, HazyTotal=840
val: GT=40, depth_npy=40, HazyTotal=120
test: GT=80, depth_npy=80, HazyTotal=240


## 4) 產生 DehazeDDPM 專用測試集（沿用 `splits/test` 檔名）

規則：

- 以 `data/splits/test/gt` 的檔案為基準（沿用既有檔名；含同目錄下 depth npy）
- 霧氣等級輸出名稱使用 `low / medium / heavy`
- 實際霧圖來源資料夾對應為：`low -> light`、`medium -> medium`、`heavy -> heavy`
- `gt_test` 與 `hazy_test` 檔名必須一致
- 80 張會做「平均分配後再隨機打散」：三種等級數量盡量平均，且可重現
- manifest 中每筆會附 `beta`（對應抽到的霧等級）與 **test split 內** depth npy 路徑

In [5]:
# 以固定種子建立可重現的等級分配
# 目標：三種霧氣程度（low / medium / heavy）數量盡量平均

dehaze_rng = random.Random(SEED + 1)

# 顯示名稱與實際資料夾名稱對應
# low 對應到原始資料夾 light
level_name_map = {
    "low": "light",
    "medium": "medium",
    "heavy": "heavy",
}
display_levels = list(level_name_map.keys())

_ensure_clean_dir(DEHAZED_TEST_ROOT, clean=CLEAN_OUTPUT_DIRS)
gt_test_dir = DEHAZED_TEST_ROOT / "gt_test"
hazy_test_dir = DEHAZED_TEST_ROOT / "hazy_test"
gt_test_dir.mkdir(parents=True, exist_ok=True)
hazy_test_dir.mkdir(parents=True, exist_ok=True)

# 沿用 splits/test/gt 的檔案與命名基準
split_test_gt_dir = SPLITS_ROOT / "test" / "gt"
split_test_gt_files = sorted([p for p in split_test_gt_dir.iterdir() if _is_image(p)])

if len(split_test_gt_files) != len(test_ids):
    raise ValueError(
        f"splits/test/gt 檔案數量與 test_ids 不一致：{len(split_test_gt_files)} vs {len(test_ids)}"
    )

# 建立「平均分配」的等級清單，再以固定種子打散
n_samples = len(split_test_gt_files)
base = n_samples // len(display_levels)
remainder = n_samples % len(display_levels)

balanced_levels: list[str] = []
for i, lvl in enumerate(display_levels):
    cnt = base + (1 if i < remainder else 0)
    balanced_levels.extend([lvl] * cnt)

dehaze_rng.shuffle(balanced_levels)

# manifest：記錄每張圖抽到的霧氣等級與來源
dehazeddpm_manifest: dict[str, object] = {
    "seed": SEED,
    "selection_seed": SEED + 1,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_split": "test",
    "source_gt_dir": str(split_test_gt_dir),
    "beta_by_fog_level": dict(BETA_MAP),
    "total_pairs": n_samples,
    "pairs": [],
    "level_hist": {lvl: 0 for lvl in display_levels},
    "level_name_map": level_name_map,
}

for idx, gt_src in enumerate(split_test_gt_files, start=1):
    sample_id = gt_src.stem

    display_level = balanced_levels[idx - 1]      # low / medium / heavy
    source_level = level_name_map[display_level]  # light / medium / heavy
    dehazeddpm_manifest["level_hist"][display_level] += 1

    hazy_src = fog_by_level_and_id[source_level][sample_id]
    depth_in_split_test = SPLITS_ROOT / "test" / "gt" / f"{sample_id}_raw_depth_meter.npy"

    # 沿用 splits/test/gt 原本檔名，再加上霧氣程度字尾
    # 例：0001.jpg -> 0001_low.jpg
    out_name = f"{gt_src.stem}_{display_level}{gt_src.suffix.lower()}"
    gt_dst = gt_test_dir / out_name
    hazy_dst = hazy_test_dir / out_name

    _copy_file(gt_src, gt_dst)
    _copy_file(hazy_src, hazy_dst)

    dehazeddpm_manifest["pairs"].append(
        {
            "index": idx,
            "id": sample_id,
            "chosen_level": display_level,
            "source_level": source_level,
            "beta": BETA_MAP[source_level],
            "depth": str(depth_in_split_test),
            "output_name": out_name,
            "gt": str(gt_dst),
            "hazy": str(hazy_dst),
            "gt_source": str(gt_src),
            "hazy_source": str(hazy_src),
        }
    )

print("=== DehazeDDPM 專用測試集建立完成 ===")
print(f"輸出位置：{DEHAZED_TEST_ROOT}")
print(f"配對數量：{dehazeddpm_manifest['total_pairs']}")
print(f"等級分佈（low/medium/heavy）：{dehazeddpm_manifest['level_hist']}")

=== DehazeDDPM 專用測試集建立完成 ===
輸出位置：dehazeddpm_test
配對數量：80
等級分佈（low/medium/heavy）：{'low': 27, 'medium': 27, 'heavy': 26}


## 5) 輸出 manifest（可追溯、可重現）

會輸出兩份 JSON：

- `data/splits/split_manifest_seed{SEED}.json`
- `data/dehazeddpm_test/dehazeddpm_manifest_seed{SEED}.json`

In [6]:
split_manifest_path = SPLITS_ROOT / f"split_manifest_seed{SEED}.json"
dehazeddpm_manifest_path = DEHAZED_TEST_ROOT / f"dehazeddpm_manifest_seed{SEED}.json"

with split_manifest_path.open("w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=2)

with dehazeddpm_manifest_path.open("w", encoding="utf-8") as f:
    json.dump(dehazeddpm_manifest, f, ensure_ascii=False, indent=2)

print("=== Manifest 輸出完成 ===")
print(f"Split manifest：{split_manifest_path}")
print(f"DehazeDDPM manifest：{dehazeddpm_manifest_path}")

=== Manifest 輸出完成 ===
Split manifest：splits/split_manifest_seed42.json
DehazeDDPM manifest：dehazeddpm_test/dehazeddpm_manifest_seed42.json


## 6) 最終統計與斷言檢查

這段會檢查：

- train/val/test 的 GT 數量是否為 280/40/80
- 每個 split 的 `gt` 目錄底下 `*_raw_depth_meter.npy` 是否與 GT 張數一致
- 每個 split 的三種霧圖數量是否分別相同
- `dehazeddpm_test` 的 `gt_test` 與 `hazy_test` 是否都是 80 張

In [7]:
# 計算某資料夾下影像總數

def _count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob("*") if _is_image(p))


def _count_depth_npy_in_gt(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.glob("*_raw_depth_meter.npy") if p.is_file())


summary_rows = []
for split_name in ("train", "val", "test"):
    gt_count = _count_images(SPLITS_ROOT / split_name / "gt")
    depth_npy_count = _count_depth_npy_in_gt(SPLITS_ROOT / split_name / "gt")
    hazy_counts = {
        level: _count_images(SPLITS_ROOT / split_name / "hazy" / level)
        for level in FOG_LEVELS
    }
    hazy_total = sum(hazy_counts.values())
    summary_rows.append((split_name, gt_count, depth_npy_count, hazy_total, hazy_counts))

print("=== 主資料集統計 ===")
for split_name, gt_count, depth_npy_count, hazy_total, hazy_counts in summary_rows:
    print(
        f"{split_name:5s} | GT={gt_count:3d} | depth_npy={depth_npy_count:3d} | "
        f"HazyTotal={hazy_total:3d} | by_level={hazy_counts}"
    )

# 核心斷言：GT 數量需符合 280 / 40 / 80
assert _count_images(SPLITS_ROOT / "train" / "gt") == N_TRAIN
assert _count_images(SPLITS_ROOT / "val" / "gt") == N_VAL
assert _count_images(SPLITS_ROOT / "test" / "gt") == N_TEST

for split_name, expected in (("train", N_TRAIN), ("val", N_VAL), ("test", N_TEST)):
    assert _count_depth_npy_in_gt(SPLITS_ROOT / split_name / "gt") == expected, (
        f"{split_name}/gt depth npy 數量錯誤"
    )

# 核心斷言：每個 split 的三種霧圖數量都應等於該 split 的 GT 數量
for split_name, expected in (("train", N_TRAIN), ("val", N_VAL), ("test", N_TEST)):
    for level in FOG_LEVELS:
        actual = _count_images(SPLITS_ROOT / split_name / "hazy" / level)
        assert actual == expected, f"{split_name}/{level} 數量錯誤：{actual} != {expected}"

# DehazeDDPM 專用測試集檢查
gt_test_count = _count_images(DEHAZED_TEST_ROOT / "gt_test")
hazy_test_count = _count_images(DEHAZED_TEST_ROOT / "hazy_test")
assert gt_test_count == N_TEST, f"gt_test 數量錯誤：{gt_test_count} != {N_TEST}"
assert hazy_test_count == N_TEST, f"hazy_test 數量錯誤：{hazy_test_count} != {N_TEST}"

print("=== DehazeDDPM 測試集統計 ===")
print(f"gt_test 數量：  {gt_test_count}")
print(f"hazy_test 數量：{hazy_test_count}")
print(f"等級分佈：{dehazeddpm_manifest['level_hist']}")
print("所有檢查皆通過。")

=== 主資料集統計 ===
train | GT=280 | depth_npy=280 | HazyTotal=840 | by_level={'light': 280, 'medium': 280, 'heavy': 280}
val   | GT= 40 | depth_npy= 40 | HazyTotal=120 | by_level={'light': 40, 'medium': 40, 'heavy': 40}
test  | GT= 80 | depth_npy= 80 | HazyTotal=240 | by_level={'light': 80, 'medium': 80, 'heavy': 80}
=== DehazeDDPM 測試集統計 ===
gt_test 數量：  80
hazy_test 數量：80
等級分佈：{'low': 27, 'medium': 27, 'heavy': 26}
所有檢查皆通過。
